# 🚀 Fine-Tuning Weave Router (Llama 3.2 3B) with Unsloth & QLoRA

This notebook trains your custom open-weights model on your `dataset.jsonl`:
- **Runtime:** Free Google Colab T4 GPU (Runtime -> Change runtime type -> T4 GPU)
- **Training Time:** ~10-15 minutes
- **Export:** Automatically exports `weave-llama3.2-3b-q4.gguf` for Ollama / local deployment.

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install xformers "transformers==4.45.2" datasets

### 1. Load Dataset
Automatically fetches `dataset.jsonl` from your repo, or lets you upload it manually.

In [ ]:
from google.colab import files
import os

if not os.path.exists('dataset.jsonl'):
    !wget -q https://raw.githubusercontent.com/darthnihilus01/weave/main/training/dataset.jsonl
    if not os.path.exists('dataset.jsonl'):
        print("Please upload your dataset.jsonl:")
        uploaded = files.upload()
    else:
        print("dataset.jsonl fetched directly from GitHub!")
else:
    print("dataset.jsonl found!")

### 2. Load Base Model in 4-bit (Llama 3.2 3B)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

### 3. Load Dataset & Train

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = load_dataset("json", data_files="dataset.jsonl", split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=120,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none"
    ),
)

trainer.train()

### 4. Test the Model

In [ ]:
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content": "You are Weave, a chill dev assistant on WhatsApp. Reply direct and lowkey with no emojis, or output JSON for tasks."},
    {"role": "user", "content": "yo weave u up"}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=100, use_cache=True)
print(tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0])

### 5. Export to GGUF format for Ollama / Local Running
This creates a quantized `.gguf` file ready to run on any computer with Ollama.

In [ ]:
# Export to 4-bit GGUF (takes ~2-3 minutes)
model.save_pretrained_gguf("weave-llama3.2-3b-q4", tokenizer, quantization_method="q4_k_m")

# Download the file to your machine
from google.colab import files
import glob
gguf_files = glob.glob("weave-llama3.2-3b-q4/*.gguf")
if gguf_files:
    print("Downloading:", gguf_files[0])
    files.download(gguf_files[0])